# 02 — Baseline Model (ResNet50)

Trains the paper baseline (ResNet50, 89.57% accuracy) on the ceilometer backscatter dataset,
with training tracked on **Weights & Biases**.

Runs both locally and on Google Colab — the dataset path comes from
`configs/config.yaml` (`dataset.path`), so update that file to point at wherever
the dataset actually lives (a local folder or Google Drive).

In [ ]:
import os, sys

IS_COLAB = os.path.exists('/content')

if IS_COLAB:
    # Mount Google Drive PRIMA di tutto
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Poi setup Colab
    repo_path = '/content/Cloud_detection_project'
    if not os.path.exists(repo_path):
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        os.system(f'git clone https://{token}@github.com/caMatt99/Cloud_detection_project.git {repo_path}')

    # Sincronizza il repository con l'ultimo codice da GitHub
    os.system('cd /content/Cloud_detection_project && git pull origin main')
    print("✓ Repository sincronizzato con GitHub")

    sys.path.insert(0, f'{repo_path}/src')
    os.chdir(repo_path)
else:
    # Setup locale
    project_root = '/Users/matteo/Desktop/Cloud_detection_project'
    os.chdir(project_root)

    # Sincronizza il repository con l'ultimo codice da GitHub
    os.system('cd /Users/matteo/Desktop/Cloud_detection_project && git pull origin main')
    print("✓ Repository sincronizzato con GitHub")

    sys.path.insert(0, f'{project_root}/src')

print(f"✓ Working dir: {os.getcwd()}")

In [ ]:
import yaml
import torch.nn as nn
import torch.optim as optim
import wandb

from dataset import get_dataloaders
from models import get_model, get_device, count_parameters, apply_dropout
from evaluate import get_predictions, compute_metrics
from engine import train_one_epoch, evaluate_one_epoch, log_epoch_to_wandb, compute_class_weights

In [3]:
if IS_COLAB:
    from google.colab import userdata
    wandb.login(key=userdata.get('WANDB_API_KEY'))
else:
    wandb.login()  # Usa ~/.netrc locale

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/matteo/.netrc.
wandb: Currently logged in as: matteo-calabretta99 (matteo-calabretta99-universit-catania) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
config_path = os.path.join(os.getcwd(), 'configs', 'config.yaml')

if not os.path.exists(config_path):
    print(f"✗ Config non trovato: {config_path}")
    print(f"  Working dir: {os.getcwd()}")
    print(f"  Files: {os.listdir(os.getcwd())}")
    raise FileNotFoundError(config_path)

with open(config_path) as f:
    cfg = yaml.safe_load(f)

env = "colab" if os.path.exists('/content') else "local"

print(f"✓ Config caricato da: {config_path}")

✓ Config caricato da: /Users/matteo/Desktop/Cloud_detection_project/configs/config.yaml


In [ ]:
config_dict = {
    "model_name":    "convnext_base",  # baseline — overrides config.yaml's model.name
    "dataset_path":  cfg["dataset"]["path"][env],
    "batch_size":    cfg["dataset"]["batch_size"],
    "image_size":    cfg["dataset"]["image_size"],
    "num_workers":   2,
    "num_classes":   cfg["model"]["num_classes"],
    "epochs":        cfg["training"]["epochs"],
    "optimizer":     cfg["training"]["optimizer"],
    "lr":            cfg["training"]["learning_rate"],
    "momentum":      cfg["training"]["momentum"],
    "weight_decay":  cfg["training"]["weight_decay"],
    "dropout_rate":       cfg["model"].get("dropout_rate", 0.3),
    "use_class_weights":  cfg["training"].get("use_class_weights", False),
    "augmentation":       cfg.get("augmentation", {}),
}

config_dict["lr"] = 0.0005  # overwrite learning rate

run = wandb.init(
    project="cloud-detection",
    name="exp-convnext-lr5e4_early_stopping",
    job_type="train",
    config=config_dict,
)

In [ ]:
device = get_device()

train_loader, val_loader, test_loader, class_names = get_dataloaders(
    dataset_path=config_dict["dataset_path"],
    batch_size=config_dict["batch_size"],
    image_size=config_dict["image_size"],
    num_workers=config_dict["num_workers"],
    aug_cfg=config_dict["augmentation"],
)

model = get_model(
    config_dict["model_name"], num_classes=config_dict["num_classes"], pretrained=True
).to(device)
model = apply_dropout(model, config_dict["model_name"], config_dict["dropout_rate"])
count_parameters(model)

run.watch(model, log="all", log_freq=10)

In [ ]:
if config_dict["use_class_weights"]:
    class_weights = compute_class_weights(train_loader.dataset).to(device)
else:
    class_weights = None

criterion = nn.CrossEntropyLoss(weight=class_weights)

if config_dict["optimizer"] == "sgd":
    optimizer = optim.SGD(
        model.parameters(),
        lr=config_dict["lr"],
        momentum=config_dict["momentum"],
        weight_decay=config_dict["weight_decay"],
    )
else:
    optimizer = optim.Adam(
        model.parameters(), lr=config_dict["lr"], weight_decay=config_dict["weight_decay"]
    )

## Training loop

`train_one_epoch` / `evaluate_one_epoch` / `log_epoch_to_wandb` live in `src/engine.py`
(shared with `03_new_model.ipynb`). Each epoch logs `train/loss`, `train/accuracy`,
`val/loss`, `val/accuracy`, and the validation confusion matrix to W&B.

In [ ]:
patience = 15
best_val_acc = 0
patience_counter = 0
delta = 0.0005

for epoch in range(1, config_dict["epochs"] + 1):
    train_loss, train_acc, _, _ = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    val_loss, val_acc, val_preds, val_labels = evaluate_one_epoch(
        model, val_loader, criterion, device
    )

    log_epoch_to_wandb(
        epoch, train_loss, train_acc, val_loss, val_acc, val_preds, val_labels, class_names
    )

    # Early stopping with delta threshold
    if val_acc > best_val_acc + delta:
        best_val_acc = val_acc
        patience_counter = 0

    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch} (best val_acc: {best_val_acc:.4f})")
        break

    print(
        f"Epoch {epoch:3d}/{config_dict['epochs']}  "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f}  "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}  "
        f"patience: {patience_counter}/{patience}"
    )

## Final evaluation on the test set

In [ ]:
test_preds, test_labels = get_predictions(model, test_loader, device)
test_metrics = compute_metrics(test_preds, test_labels, model_name=config_dict["model_name"])

wandb.log(
    {
        "test/accuracy": test_metrics["accuracy"],
        "test/f1": test_metrics["f1"],
        "test/precision": test_metrics["precision"],
        "test/recall": test_metrics["recall"],
        "test/confusion_matrix": wandb.plot.confusion_matrix(
            preds=test_preds,
            y_true=test_labels,
            class_names=class_names,
        ),
    }
)

## Decision threshold tuning

Default threshold is 0.5, but with an imbalanced train set (see class weighting above)
it's not guaranteed to be optimal. Finds the threshold that maximizes F1 on the
validation set, then compares test accuracy at 0.5 vs. that tuned threshold.

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve, accuracy_score

from evaluate import get_probabilities

# Probabilita' softmax della classe "true" (cloud), su validation e test set.
val_probs, val_labels_arr = get_probabilities(model, val_loader, device)
test_probs, test_labels_arr = get_probabilities(model, test_loader, device)

# Soglia che massimizza l'F1 sul validation set (thresholds ha un elemento in
# meno di precision/recall — l'ultimo punto di precision/recall non ha soglia).
precision, recall, thresholds = precision_recall_curve(val_labels_arr, val_probs)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-6)
best_threshold = thresholds[np.argmax(f1_scores[:-1])]

print(f"Default threshold: 0.500")
print(f"Optimal threshold:  {best_threshold:.3f}")

test_preds_default = (test_probs >= 0.5).astype(int)
test_preds_tuned = (test_probs >= best_threshold).astype(int)

acc_default = accuracy_score(test_labels_arr, test_preds_default)
acc_tuned = accuracy_score(test_labels_arr, test_preds_tuned)

print(f"Test accuracy @0.5:     {acc_default:.4f}")
print(f"Test accuracy @optimal: {acc_tuned:.4f}")

wandb.log({
    "test/accuracy_threshold_0.5": acc_default,
    "test/accuracy_threshold_optimal": acc_tuned,
})

## Save checkpoint and log it as a W&B artifact

In [ ]:
import torch

checkpoint_dir = os.path.join(os.getcwd(), "checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, f"{config_dict['model_name']}.pt")
torch.save(model.state_dict(), checkpoint_path)

artifact = wandb.Artifact(
    name=f"{config_dict['model_name']}-checkpoint",
    type="model",
    metadata=dict(test_metrics),
)
artifact.add_file(checkpoint_path)
run.log_artifact(artifact)

run.finish()